In [23]:
import pandas as pd

In [24]:
df = pd.read_csv(
    "openpowerlifting-2024-01-06-4c732975.csv",low_memory=False
)
df.head()

,Name,Sex,Event,Equipment,Age,AgeClass,BirthYearClass,Division,BodyweightKg,WeightClassKg,...,Tested,Country,State,Federation,ParentFederation,Date,MeetCountry,MeetState,MeetTown,MeetName
0,Alona Vladi,F,SBD,Raw,33.0,24-34,24-39,O,58.30,60,...,Yes,Russia,NaN,GFP,NaN,2019-05-11,Russia,NaN,Bryansk,Open Tournament
1,Galina Solovyanova,F,SBD,Raw,43.0,40-44,40-49,M1,73.10,75,...,Yes,Russia,NaN,GFP,NaN,2019-05-11,Russia,NaN,Bryansk,Open Tournament
2,Daniil Voronin,M,SBD,Raw,15.5,16-17,14-18,T,67.40,75,...,Yes,Russia,NaN,GFP,NaN,2019-05-11,Russia,NaN,Bryansk,Open Tournament
3,Aleksey Krasov,M,SBD,Raw,35.0,35-39,24-39,O,66.65,75,...,Yes,Russia,NaN,GFP,NaN,2019-05-11,Russia,NaN,Bryansk,Open Tournament
4,Margarita Pleschenkova,M,SBD,Raw,26.5,24-34,24-39,O,72.45,75,...,Yes,Russia,NaN,GFP,NaN,2019-05-11,Russia,NaN,Bryansk,Open Tournament


##### Reviewing dataset structure 
- What's in the file? Checking column names, data types, missingness, etc

In [25]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043013 entries, 0 to 3043012
Data columns (total 41 columns):
 #   Column            Dtype  
---  ------            -----  
 0   Name              object 
 1   Sex               object 
 2   Event             object 
 3   Equipment         object 
 4   Age               float64
 5   AgeClass          object 
 6   BirthYearClass    object 
 7   Division          object 
 8   BodyweightKg      float64
 9   WeightClassKg     object 
 10  Squat1Kg          float64
 11  Squat2Kg          float64
 12  Squat3Kg          float64
 13  Squat4Kg          float64
 14  Best3SquatKg      float64
 15  Bench1Kg          float64
 16  Bench2Kg          float64
 17  Bench3Kg          float64
 18  Bench4Kg          float64
 19  Best3BenchKg      float64
 20  Deadlift1Kg       float64
 21  Deadlift2Kg       float64
 22  Deadlift3Kg       float64
 23  Deadlift4Kg       float64
 24  Best3DeadliftKg   float64
 25  TotalKg           float64
 26  Place         

In [26]:
df.isna().sum()

Name                      0
Sex                       0
Event                     0
Equipment                 0
Age                 1070337
AgeClass             814256
BirthYearClass      1000666
Division               1435
BodyweightKg          38368
WeightClassKg         40757
Squat1Kg            2137906
Squat2Kg            2147142
Squat3Kg            2171053
Squat4Kg            3034560
Best3SquatKg        1009169
Bench1Kg            1652910
Bench2Kg            1668528
Bench3Kg            1710945
Bench4Kg            3022026
Best3BenchKg         358566
Deadlift1Kg         2021904
Deadlift2Kg         2039602
Deadlift3Kg         2080157
Deadlift4Kg         3020077
Best3DeadliftKg      836823
TotalKg              202421
Place                     0
Dots                 227397
Wilks                227397
Glossbrenner         227397
Goodlift             476889
Tested               804298
Country             1284501
State               2408023
Federation                0
ParentFederation    

##### Data quality checks and data cleaning


In [27]:
# are there duplicate rows?

df_clean = df.drop_duplicates().copy()

orig_count = len(df)
clean_count = len(df_clean)
print(f"Original df row count: {orig_count}")
print(f"Clean df row count: {clean_count}")
print(f"Duplicates removed: {orig_count - clean_count}")

Original df row count: 3043013
Clean df row count: 3039412
Duplicates removed: 3601


Finding: 3,601 exact duplicate rows (about 0.1%). Removed them and df_clean is used through the rest of the notebook.

In [28]:
# why is TotalKg missing for some lifters?

df_clean.loc[df_clean["TotalKg"].isna(), "Place"].value_counts()

Place
DQ    184378
NS     16023
DD      1660
Name: count, dtype: int64

Finding: Every missing TotalKg has Place = DQ (disqualified), NS (no show), or DD (doping disqualification). 

This is a nuance of the data, not a data quality issue. When analyzing totals, drop missing TotalKG. When analyzing bombouts, keep missing total where place = DQ.

In [29]:
#why is bodyweight missing for some lifters?

bw_missing = df_clean["BodyweightKg"].isna()
class_missing = df_clean["WeightClassKg"].isna()

print(f"Missing bodyweight: {bw_missing.sum()} rows ({bw_missing.sum() / len(df_clean) * 100:.2f}%)")
print(f"  - and weight class is known: {(bw_missing & ~class_missing).sum()}")
print(f"  - and weight class is also missing: {(bw_missing & class_missing).sum()}")

# Which federations are missing it most?
df_clean.loc[bw_missing, "Federation"].value_counts().head()

Missing bodyweight: 38346 rows (1.26%)
  - and weight class is known: 19539
  - and weight class is also missing: 18807


Federation
USPF      8907
THSPA     6937
USAPL     2734
THSWPA    2636
WABDL     2456
Name: count, dtype: int64

Finding: There are 38,346 rows missing bodyweight. About half of them still have a weight class, and the other half have neither. 

Decision: Leave bodyweight missing instead of filling it from the weight class. A weight class is only an upper limit, someone in the 75 kg class could weigh 68 kg. Drop these rows only for analyses that need exact bodyweight, like bodyweight vs total.

In [30]:
# is bodyweight within the weight class?

weight_class = df_clean["WeightClassKg"].astype(str)
is_plus_class = weight_class.str.endswith("+")
class_limit = pd.to_numeric(weight_class.str.rstrip("+"), errors="coerce")

has_both = df_clean["BodyweightKg"].notna() & class_limit.notna()

# Regular class: bodyweight should be at or under the limit

over_limit = has_both & ~is_plus_class & (df_clean["BodyweightKg"] > class_limit)

# super heavy class: bodyweight should be over the limit
under_plus = has_both & is_plus_class & (df_clean["BodyweightKg"] <= class_limit)

print(f"Over their class limit: {over_limit.sum()}")
print(f"In a + class but not over the limit: {under_plus.sum()}")
print(f"Weight class is just '+' (no number): {(weight_class == '+').sum()}")

# How far over the limit are they?
(df_clean["BodyweightKg"] - class_limit)[over_limit].describe()

Over their class limit: 29678
In a + class but not over the limit: 0
Weight class is just '+' (no number): 475


count    29678.000000
mean         0.046088
std          0.028047
min          0.010000
25%          0.020000
50%          0.040000
75%          0.070000
max          0.090000
dtype: float64

In [31]:
# Which federations have the most lifters over their class limit?
df_clean.loc[over_limit, "Federation"].value_counts().head(10)

Federation
THSPA       22950
THSWPA       3979
USA-UA       1214
WNPF          642
MHSPLA        256
USPF          190
FHSAA         145
USSF           81
Hardcore       50
LHSPLA         44
Name: count, dtype: int64

Finding: 29,678 lifters are over their class limit, but  all by less than 0.1 kg. This is likely due to rounding from pound-based classes converted to kg, mostly Texas high school meets (THSPA, THSWPA). 

**Decision:** No real problems, so no changes.

In [32]:
# does TotalKg equal best squat + bench + deadlift?
# only SBD events are check here since other types of events may not have all 3 lifts

sbd_totals = df_clean[(df_clean["Event"] == "SBD") & df_clean["TotalKg"].notna()]

# checking that min_count=3 because the sum is NaN unless all three lifts are recorded

lift_sum = sbd_totals[["Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg"]].sum(axis=1, min_count=3)
missing_a_lift = lift_sum.isna()
mismatch = (sbd_totals["TotalKg"] - lift_sum).abs() > 0.5

print(f"SBD entries with a total: {len(sbd_totals)}")
print(f"Total doesn't match the three lifts: {mismatch.sum()}")
print(f"Has a total but individual lifts are missing: {missing_a_lift.sum()}")

# Which federations only record totals?
sbd_totals.loc[missing_a_lift, "Federation"].value_counts().head()

SBD entries with a total: 1951221
Total doesn't match the three lifts: 0
Has a total but individual lifts are missing: 11089


Federation
USPF      3634
USA-UA    1924
THSPA      758
LHSPLA     626
MHSPLA     478
Name: count, dtype: int64

Finding: Every total matches the sum of the three best lifts. 11,089 SBD entries have a total but no individual lifts. 

Decision: Keep them for analyses of totals. Drop them for analyses of individual lifts, like lift correlations or predicting deadlift from squat.

In [33]:
# does a negative attempt mean a missed lift?

for lift in ["Squat", "Bench", "Deadlift"]:
    negative_best = df_clean[f"Best3{lift}Kg"] < 0
    with_total = negative_best & df_clean["TotalKg"].notna()
    print(f"{lift}: {negative_best.sum()} negative best lifts, {with_total.sum()} of them have a total")

# Place for lifters whose best squat is negative
df_clean.loc[df_clean["Best3SquatKg"] < 0, "Place"].value_counts()

Squat: 3020 negative best lifts, 0 of them have a total
Bench: 3124 negative best lifts, 0 of them have a total
Deadlift: 1185 negative best lifts, 0 of them have a total


Place
DQ    3018
DD       1
NS       1
Name: count, dtype: int64

Finding: A negative best lift means the lifter missed all three attempts. None of them have a total, and almost all have Place = DQ. This confirms that negative = missed.

No change needed. Filtering on TotalKg.notna() already removes these rows.

In [34]:
# do 4th attempts count toward the total?

for lift in ["Squat", "Bench", "Deadlift"]:
    fourth = df_clean[f"{lift}4Kg"]
    heavier = (fourth > 0) & (fourth > df_clean[f"Best3{lift}Kg"])
    print(f"{lift}: {fourth.notna().sum()} 4th attempts, {heavier.sum()} heavier than the best of 3")

Squat: 8432 4th attempts, 6198 heavier than the best of 3
Bench: 20941 4th attempts, 12607 heavier than the best of 3
Deadlift: 22876 4th attempts, 17043 heavier than the best of 3


Finding: 4th attempts are rare (under 1% of rows), and most successful ones are heavier than the lifter's best of 3. Some federations allow 4th attempts if the attempt is a record.

Source: https://apopowerlifting.com/wp-content/uploads/2025/12/APO-Rules-YGD-09.24.2025-v1.14b.pdf

The previous check determined that the total is the sum of the best of the first three attempts, so 4th attempts aren't included in the Total.

Decision: Use the 'Best3' columns and ignore the 4th-attempt columns.


##### Cleaning summary
- Duplicate rows
    - 3,601 found 
    - Removed
- Missing TotalKg 
    - All bomb-outs, no-shows, or doping DQs
    - Kept - filter based on relevant analysis
- Missing bodyweight 
    - About 1.3% of rows 
    - Left missing - not filled from weight class 
- Bodyweight vs weight class 
    - Only rounding (< 0.1 kg) 
    -  No change
- TotalKg = sum of best lifts
    - Always matches
    - No change; about 0.6% of SBD rows have totals only
Negative attempts
    - Missed lifts
    -  No change
- 4th attempts
    - Record attempts, not in total
    - Use 'Best3' columns only |